In [ ]:
import os
import sys
import shutil
import tarfile
import torch
import gc
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from torchvision import transforms
from google.colab import drive
from tqdm import tqdm


if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')


print("Installing libraries...")
!pip install -q git+https://github.com/Cadene/pretrained-models.pytorch
!pip install -q timm einops pretrainedmodels


if not os.path.exists("action-models"):
    !git clone -q https://github.com/epic-kitchens/action-models
    print("Repository cloned.")


repo_path = Path("action-models")
temporal_shift_path = repo_path / "ops" / "temporal_shift.py"
tsm_path = repo_path / "tsm.py"


if temporal_shift_path.exists():
    text = temporal_shift_path.read_text()
    old_code = "(torchvision.models.ResNet, pretrainedmodels.models.torchvision_models.ResNet)"
    new_code = "(torchvision.models.resnet.ResNet,)"
    if old_code in text:
        text = text.replace(old_code, new_code)
        temporal_shift_path.write_text(text)
        print("Patch applied: temporal_shift.py")

if tsm_path.exists():
    text = tsm_path.read_text()
    old_view = "base_model_input = input.view((-1, sample_len) + input.size()[-2:])"
    new_reshape = "base_model_input = input.reshape((-1, sample_len) + input.size()[-2:])"
    if old_view in text:
        text = text.replace(old_view, new_reshape)
        tsm_path.write_text(text)
        print("Patch applied: tsm.py")

Mounted at /content/drive
Installing libraries...
  Preparing metadata (setup.py) ... done
Repository cloned.
Patch applied: temporal_shift.py
Patch applied: tsm.py


In [ ]:
TAR_DIR = "/content/drive/MyDrive/EPIC-KITCHENS55/P14"
FRAMES_ROOT = "/content/P14_Videos"
ANNOTATION_PATH = "/content/drive/MyDrive/dl_project/epic-kitchens-55-annotations"
EXTRACTED_FPS = 30
NUM_SEGMENTS = 8
NUM_VIEWS = 10
device = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(FRAMES_ROOT, exist_ok=True)
if len(os.listdir(FRAMES_ROOT)) == 0:
    print("Extracting videos (This might take a while)...")
    for tar_file in os.listdir(TAR_DIR):
        if tar_file.endswith(".tar"):
            video_id = tar_file.replace(".tar","")
            out_dir = os.path.join(FRAMES_ROOT, video_id)
            os.makedirs(out_dir, exist_ok=True)
            with tarfile.open(os.path.join(TAR_DIR, tar_file)) as tar:
                tar.extractall(path=out_dir)
            print(f"Extracted: {video_id}")


Extracting videos (This might take a while)...


/tmp/ipython-input-1429123562.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=out_dir)


Extracted: P14_01
Extracted: P14_02
Extracted: P14_03
Extracted: P14_04
Extracted: P14_05
Extracted: P14_07
Extracted: P14_09


In [ ]:

try:
    verb_df = pd.read_csv(f"{ANNOTATION_PATH}/EPIC_verb_classes.csv")
    noun_df = pd.read_csv(f"{ANNOTATION_PATH}/EPIC_noun_classes.csv")
    label_df = pd.read_csv(f"{ANNOTATION_PATH}/EPIC_train_action_labels.csv")
    video_info = pd.read_csv(f"{ANNOTATION_PATH}/EPIC_video_info.csv")

    verb_map = dict(zip(verb_df.verb_id, verb_df.class_key))
    noun_map = dict(zip(noun_df.noun_id, noun_df.class_key))
    fps_map = dict(zip(video_info.video, video_info.fps))
except Exception as e:
    print(f"ERROR: Annotation files not found! Path: {ANNOTATION_PATH}\nError Detail: {e}")

preprocess_test = transforms.Compose([
    transforms.Resize(256),
    transforms.FiveCrop(224),
    transforms.Lambda(lambda crops: torch.stack([
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])(
            transforms.ToTensor()(c)
        ) for c in crops[:3]
    ]))
])

def get_multi_view_clip(video_id, start_frame, stop_frame):
    start_frame, stop_frame = int(start_frame), int(stop_frame)
    if video_id not in fps_map: return None

    real_fps = float(fps_map[video_id])
    scale = EXTRACTED_FPS / real_fps

    new_start = int(start_frame * scale)
    new_stop  = int(stop_frame  * scale)

    if new_stop <= new_start + NUM_SEGMENTS:
        new_stop = new_start + NUM_SEGMENTS + 1

    clip_length = new_stop - new_start



    all_views = []


    seg_size = float(clip_length) / NUM_SEGMENTS

    for i in range(NUM_VIEWS):
        frac_indices = np.linspace(new_start, new_stop - 1, NUM_SEGMENTS)
        frame_ids = frac_indices.astype(int)
        frames = []
        for idx in frame_ids:
            path = f"{FRAMES_ROOT}/{video_id}/frame_{str(idx).zfill(10)}.jpg"
            if not os.path.exists(path):
                continue

            try:
                img = Image.open(path).convert("RGB")
                crops = preprocess_test(img) # [3, 3, 224, 224] (3 crop)
                frames.append(crops)
            except:
                continue
        if len(frames) < NUM_SEGMENTS:
            continue

        clip_stack = torch.stack(frames, dim=1)
        all_views.append(clip_stack)

    if len(all_views) == 0:
        raise RuntimeError(f"No valid frames for {video_id}")

    clip = torch.cat(all_views, dim=0) # [N_actual * 3, 8, 3, 224, 224]
    return clip.to(device)

In [ ]:
import torch
import gc
import pandas as pd
import numpy as np


ARCHITECTURES = ["TSM","TRN", "MTRN","TSN"]
BASE_MODEL = "resnet50"
NUM_SEGMENTS = 8

label_df_p14_head = label_df[label_df['participant_id'] == "P14"].head(20)

for arch in ARCHITECTURES:

    print(f"EVALUATING: {arch}")


    if 'model' in globals():
        del model
    torch.cuda.empty_cache()
    gc.collect()


    try:
        print(f"Loading {arch}...")
        model = torch.hub.load(
            '/content/action-models',
            arch,
            (125, 352),
            NUM_SEGMENTS,
            'RGB',
            base_model=BASE_MODEL,
            pretrained='epic-kitchens',
            trust_repo=True,
            source='local'
        )
    except Exception as e:
        print(f"Error loading {arch}: {e}")
        continue

    model.to(device)
    model.eval()


    results = []

    with torch.no_grad():
        for _, row in label_df_p14_head.iterrows():
            try:

                clip = get_multi_view_clip(row.video_id, row.start_frame, row.stop_frame)
            except RuntimeError:
                continue
            except Exception as e:
                print(f"Data error for {row.video_id}: {e}")
                continue


            v_logit, n_logit = model(clip)


            v_pred = v_logit.mean(0).argmax().item()
            n_pred = n_logit.mean(0).argmax().item()

            results.append({
                "video_id": row.video_id,
                "GT_Action": f"{verb_map[row.verb_class]} {noun_map[row.noun_class]}",
                "Pred_Action": f"{verb_map.get(v_pred, '?')} {noun_map.get(n_pred, '?')}",
                "Verb_Correct": v_pred == row.verb_class,
                "Noun_Correct": n_pred == row.noun_class,
                "Action_Correct": (v_pred == row.verb_class) and (n_pred == row.noun_class)
            })


    df_res = pd.DataFrame(results)
    if not df_res.empty:

        print(df_res[["video_id", "GT_Action", "Pred_Action", "Action_Correct"]].to_string(index=False))


        print(f"\nResults for {arch}:")
        print(f"Verb Acc:   {df_res['Verb_Correct'].mean()*100:.2f}%")
        print(f"Noun Acc:   {df_res['Noun_Correct'].mean()*100:.2f}%")
        print(f"Action Acc: {df_res['Action_Correct'].mean()*100:.2f}%")
    else:
        print("No results generated.")

    del model
    torch.cuda.empty_cache()
    gc.collect()

print("\nEvaluation complete.")

EVALUATING: TSM
Loading TSM...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://data.bris.ac.uk/datasets/2tw6gdvmfj3f12papdy24flvmo/TSM_arch=resnet50_modality=RGB_segments=8-cfc93918.pth.tar" to /root/.cache/torch/hub/checkpoints/TSM_arch=resnet50_modality=RGB_segments=8-cfc93918.pth.tar


100%|██████████| 93.7M/93.7M [01:17<00:00, 1.27MB/s]


video_id             GT_Action    Pred_Action  Action_Correct
  P14_01          turn-on oven turn-on paella           False
  P14_01            fix paella   turn-on oven           False
  P14_01               put oil        put oil            True
  P14_01          close bottle        put oil           False
  P14_01            put bottle       put beef           False
  P14_01              open box        put oil           False
  P14_01              open bag     put bottle           False
  P14_01 turn-on fan:extractor       open box           False
  P14_01            take plate       take box           False
  P14_01             take fork       open box           False
  P14_01            take knife       put beef           False
  P14_01             take beef     open bread           False
  P14_01              put beef     open bread           False
  P14_01             take beef       open bag           False
  P14_01              put beef       open box           False
  P14_01

...

video_id             GT_Action       Pred_Action  Action_Correct
  P14_01          turn-on oven      turn-on oven            True
  P14_01            fix paella      turn-on oven           False
  P14_01               put oil          take oil           False
  P14_01          close bottle           put oil           False
  P14_01            put bottle           put oil           False
  P14_01              open box          take oil           False
  P14_01              open bag        put bottle           False
  P14_01 turn-on fan:extractor        take knife           False
  P14_01            take plate        take cream           False
  P14_01             take fork       take bottle           False
  P14_01            take knife        take bread           False
  P14_01             take beef take wrap:plastic           False
  P14_01              put beef        open bread           False
  P14_01             take beef        open bread           False
  P14_01              put

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://data.bris.ac.uk/datasets/2tw6gdvmfj3f12papdy24flvmo/TSN_arch=resnet50_modality=RGB_segments=8-3ecf904f.pth.tar" to /root/.cache/torch/hub/checkpoints/TSN_arch=resnet50_modality=RGB_segments=8-3ecf904f.pth.tar


100%|██████████| 93.7M/93.7M [01:18<00:00, 1.24MB/s]


video_id             GT_Action    Pred_Action  Action_Correct
  P14_01          turn-on oven turn-on paella           False
  P14_01            fix paella   turn-on oven           False
  P14_01               put oil       put beef           False
  P14_01          close bottle        put oil           False
  P14_01            put bottle        put oil           False
  P14_01              open box       put beef           False
  P14_01              open bag     put bottle           False
  P14_01 turn-on fan:extractor       open box           False
  P14_01            take plate       open box           False
  P14_01             take fork       open box           False
  P14_01            take knife       open box           False
  P14_01             take beef       open box           False
  P14_01              put beef       open box           False
  P14_01             take beef       open box           False
  P14_01              put beef       open box           False
  P14_01

In [ ]:
import torch
import gc
import csv
import json
import pandas as pd
from tqdm import tqdm
from google.colab import files

ARCHITECTURES = ["TSM","TRN", "MTRN","TSN"]
BASE_MODEL = "resnet50"
NUM_SEGMENTS = 8
TOP_K = 20

label_df_p14 = label_df[label_df['participant_id'] == "P14"]

for arch in ARCHITECTURES:
    print(f"PROCESSING: {arch}")


    if 'model' in globals():
        del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"Loading {arch}...")
    try:
        model = torch.hub.load(
            '/content/action-models',
            arch,
            (125, 352),
            NUM_SEGMENTS,
            'RGB',
            base_model=BASE_MODEL,
            pretrained='epic-kitchens',
            trust_repo=True,
            source='local'
        )
    except Exception as e:
        print(f"Error loading {arch}: {e}")
        continue

    model.to(device)
    model.eval()

    results = []

    with torch.no_grad():
        for video_id, video_rows in label_df_p14.groupby("video_id"):
            print(f"Processing {video_id} with {arch}...")

            for _, row in tqdm(video_rows.iterrows(), total=len(video_rows), desc=f"{video_id}"):
                try:
                    clip = get_multi_view_clip(row.video_id, row.start_frame, row.stop_frame)
                except RuntimeError:
                    continue
                except Exception:
                    continue

                v_logit, n_logit = model(clip)

                v_pred = v_logit.mean(0).argmax().item()
                n_pred = n_logit.mean(0).argmax().item()

                is_verb_correct = (v_pred == row.verb_class)
                is_noun_correct = (n_pred == row.noun_class)
                is_action_correct = (is_verb_correct and is_noun_correct)

                v_probs = torch.softmax(v_logit.mean(0), dim=0)
                n_probs = torch.softmax(n_logit.mean(0), dim=0)
                v_probs_cpu = v_probs.cpu().numpy()
                n_probs_cpu = n_probs.cpu().numpy()

                all_combos = []
                for vi, vp in enumerate(v_probs_cpu):
                    for ni, np_ in enumerate(n_probs_cpu):
                        prob = float(vp * np_)
                        if prob > 0.001:
                            narration = f"{verb_map[vi]} {noun_map[ni]}"
                            all_combos.append({"answer": narration, "confidence": prob})

                top20 = sorted(all_combos, key=lambda x: x["confidence"], reverse=True)[:TOP_K]

                results.append({
                    "video_id": row.video_id,
                    "start_frame": row.start_frame,
                    "stop_frame": row.stop_frame,
                    "ground_truth": f"{verb_map[row.verb_class]} {noun_map[row.noun_class]}",
                    "distractors_with_confidence": top20,
                    "num_options": TOP_K,
                    "verb_correct": is_verb_correct,
                    "noun_correct": is_noun_correct,
                    "action_correct": is_action_correct
                })

                del clip, v_logit, n_logit, v_probs, n_probs

    df_results = pd.DataFrame(results)
    print(f"\nResults for {arch} (Full Dataset):")
    if not df_results.empty:
        print(f"Verb Acc:   {df_results['verb_correct'].mean()*100:.2f}%")
        print(f"Noun Acc:   {df_results['noun_correct'].mean()*100:.2f}%")
        print(f"Action Acc: {df_results['action_correct'].mean()*100:.2f}%")
    else:
        print("No results found.")

    csv_filename = f"p14_{arch}.csv"
    print(f"Saving CSV to {csv_filename}...")

    with open(csv_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "video_id", "start_frame", "stop_frame",
            "ground_truth", "distractors_with_confidence", "num_options"
        ], extrasaction='ignore')

        writer.writeheader()

        for r in results:
            row_csv = r.copy()
            row_csv["distractors_with_confidence"] = json.dumps(row_csv["distractors_with_confidence"], ensure_ascii=False)
            writer.writerow(row_csv)

    print(f"File saved: {csv_filename}")
    files.download(csv_filename)

    print(f"Finished {arch}")
    del model, results, df_results
    torch.cuda.empty_cache()
    gc.collect()

print("ALL ARCHITECTURES PROCESSED SUCCESSFULLY!")

PROCESSING: TSM
Loading TSM...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Processing P14_01 with TSM...


P14_01: 100%|██████████| 17/17 [00:06<00:00,  2.77it/s]


Processing P14_02 with TSM...


...

P14_09: 100%|██████████| 36/36 [00:12<00:00,  2.87it/s]


Results for TSN (Full Dataset):
Verb Acc:   25.45%
Noun Acc:   20.91%
Action Acc: 7.27%
Saving CSV to p14_TSN.csv...
File saved: p14_TSN.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Finished TSN
ALL ARCHITECTURES PROCESSED SUCCESSFULLY!
